In [1]:
import numpy as np
import matplotlib.pyplot as plt
import copy

import pyccl as ccl

import treecorr

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.size": 16,          # base font size
    "axes.labelsize": 18,     # x and y labels
    "axes.titlesize": 18,     # title
    "legend.fontsize": 14,    # legend
    "xtick.labelsize": 14,    # x tick labels
    "ytick.labelsize": 14,    # y tick labels
})


import sys
import os
import copy

sys.path.append(os.path.abspath("/home/milan/Desktop/thesis/code")) 

In [ ]:
from halo_model.power_spectra.matter_power_baryons import MatterPowerBaryons
from halo_model.halos.base.profile.profile import CompositeProfile

from halo_model.config.config import Config
from halo_model.algorithms.ridders_derivative import ridders_derivative

from halo_model.power_spectra.xi_computer import compute_Pk_2d, Pk_2d_to_Cl, Cl_to_xi, compute_xi, error_xi
from halo_model.power_spectra.del_xi import del_xi

sys.path.append(os.path.abspath("/home/milan/Desktop/thesis/code/plotting_code/constraining_power")) 

In [3]:
# set config 
cfg = Config()

In [6]:
# Compute power spectrum (at various redshifts):

h_init = 0.2
h = h_init
n_iters = 3

# Shape: [n_iters, N_theta] for low and up separately
xi_lows = []
xi_ups  = []
        
for i in range(n_iters):
    cfg_low = copy.deepcopy(cfg)
    cfg_up = copy.deepcopy(cfg) 
    
    cfg_low.beta = cfg.beta - h
    cfg_up.beta = cfg.beta + h
    
    xi_up = compute_xi(cfg_up)
    xi_low = compute_xi(cfg_low)
    
    print(xi_up - xi_low)
    
    xi_lows.append(xi_low)   # shape: [N_theta]
    xi_ups.append(xi_up)
        
    h *= 0.5


Computing Fourier transform of gas profile...
Computing Fourier transform of stellar profile...
computing power spectrum at redshift 2.0
interpolating Ic and Jc functions...
Computing Fourier transform of gas profile...
Computing Fourier transform of stellar profile...
computing power spectrum at redshift 1.8
interpolating Ic and Jc functions...
Computing Fourier transform of gas profile...
Computing Fourier transform of stellar profile...
computing power spectrum at redshift 1.6
interpolating Ic and Jc functions...
Computing Fourier transform of gas profile...
Computing Fourier transform of stellar profile...
computing power spectrum at redshift 1.4
interpolating Ic and Jc functions...
Computing Fourier transform of gas profile...
Computing Fourier transform of stellar profile...
computing power spectrum at redshift 1.2
interpolating Ic and Jc functions...
Computing Fourier transform of gas profile...
Computing Fourier transform of stellar profile...
computing power spectrum at redshi

In [7]:
# pairs[i, theta_idx] = [xi_low, xi_up] at iteration i
# ridders_derivative expects pairs[:, theta_idx] -> shape [n_iters, 2]
pairs = np.array([
    [[xi_lows[i][t], xi_ups[i][t]] for t in range(cfg.N_theta)]
    for i in range(n_iters)
])  # shape: [n_iters, N_theta, 2]
   

In [ ]:

# Compute derivative for each theta angle
derivs = np.array([
    ridders_derivative(pairs[:, t, :], h_init=h_init, d=2, eps=1e-5)
    for t in range(cfg.N_theta)
])

print(derivs)   

[ 7.57600986e-05  7.44125142e-05  6.84926965e-05  4.72177465e-05
  1.60247347e-05  3.96243695e-06  1.56869787e-07 -2.12100568e-07
 -2.68351589e-08 -3.54334825e-09]


In [6]:
np.save("derivs", derivs)

In [8]:
derivs = np.load("derivs.npy")

In [14]:
errors = error_xi(cfg)
print(errors)

0.5678404104087887
[2.14404829e-05 7.42294059e-06 5.14017829e-06 2.61068372e-06
 1.49399068e-06 5.01533684e-07 3.27222240e-07 1.55436245e-07
 9.45149741e-08 3.81644652e-08]


In [16]:
F = np.sum((derivs/errors)**2)

print(np.sqrt(1/F))

0.03541512014318836
